# SIH26174 — Notebook 04: Pose / Hand Testing (Pretrained MediaPipe)

**Project:** BAS Experiment Monitor (SIH26174) — AI Human Activity Recognition for On-board BAS Experiments
**Part:** Part 1 — Model Development (Training Pipeline)
**Stage:** Notebook 04 of the training pipeline

## 1. Objective and Scope

This notebook tests whether **pretrained** pose- and hand-landmark estimation is practically usable
as a supporting visual-perception component for the SIH26174 runtime system. It is a
**diagnostic/testing** notebook, not a training notebook.

It determines:

1. Whether pretrained pose estimation loads and runs correctly on representative HMDB51 frames.
2. Whether pretrained hand estimation loads and runs correctly on the same frames.
3. How often landmarks are actually detected (a detection-availability rate, **not** an accuracy
   metric).
4. Whether the detected landmarks look visually reasonable.
5. What qualitative limitations show up (occlusion, missing hands, unusual framing, etc.).
6. Basic inference timing for both models on this small sample.

### What this notebook does NOT do

- It does **not** train a custom pose model or a custom hand model.
- It does **not** create laboratory-specific pose/hand labels.
- It does **not** implement experiment-validation, state-machine, GUI, or voice-alert logic
  (Part 2 concerns).
- It does **not** load, run, or train YOLOv8 (that is Notebook 03's job; loaded here only if a
  visualization were otherwise impossible — in practice this notebook does not need it at all).
- It does **not** load or train the future action-recognition model, and does not classify HMDB51
  actions.
- It does **not** run over the full HMDB51 dataset — only a small, deterministic sample.

### Dataset boundary

HMDB51 is an **action-recognition** dataset, not a laboratory pose/hand dataset. This notebook
only checks whether pretrained pose/hand estimation produces useful landmarks on representative
HMDB51 frames. It does **not** prove pose/hand performance on the eventual SIH laboratory
experiment, which will involve different framing, objects, and body orientations.

### Architectural boundary

```text
YOLOv8 (Notebook 03)
    |
    v
object detections

Pose / Hand (THIS notebook)
    |
    v
landmarks

Action recognition (later notebook)
    |
    v
action + confidence
```

This notebook produces landmarks only. It does not decide what action is happening and does not
validate any experiment sequence.


## 2. Imports

In [1]:
import os
import sys
import json
import time
import random
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

print("Python version:", sys.version.split()[0])
print("OpenCV version:", cv2.__version__)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)


Python version: 3.13.15
OpenCV version: 4.14.0
Pandas version: 2.2.3
NumPy version: 2.1.3


### 2.1 Install / Import MediaPipe

MediaPipe is installed if not already available. This notebook uses MediaPipe's classic
`solutions` API (`mp.solutions.pose`, `mp.solutions.hands`) — a stable, widely available pretrained
pose/hand solution that is practical to run in Google Colab without downloading separate task
bundles.


In [2]:
try:
    import mediapipe as mp
    print("MediaPipe already available:", mp.__version__)
except ImportError:
    print("MediaPipe not found — installing (this may take a minute)...")
    %pip install -q mediapipe
    import mediapipe as mp
    print("MediaPipe installed:", mp.__version__)


MediaPipe not found — installing (this may take a minute)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 10.6 MB/s eta 0:00:00
MediaPipe installed: 1.0.1


## 3. Configuration

Every path, sampling parameter, and confidence threshold used in this notebook is defined here —
nowhere else. `DATASET_ROOT` is the only hardcoded absolute path, consistent with Notebooks 01-03.


In [3]:
# ---------------------------------------------------------------------------
# Dataset location (the single hardcoded absolute path allowed in this notebook)
# ---------------------------------------------------------------------------
DATASET_ROOT = Path(
    "/content/drive/MyDrive/Technical-Projects/SIH26174/training/data/raw/hmdb51_sta"
)

# ---------------------------------------------------------------------------
# Project paths, derived from DATASET_ROOT (same convention as Notebooks 01-03)
# ---------------------------------------------------------------------------
TRAINING_ROOT = DATASET_ROOT.parents[2]
CONFIGS_DIR = TRAINING_ROOT / "configs"
REPORTS_DIR = TRAINING_ROOT / "reports"

CLASS_MAPPING_PATH = CONFIGS_DIR / "action_classes.json"  # Notebook 01 output
POSE_HAND_REPORT_JSON_PATH = REPORTS_DIR / "pose_hand_test.json"
POSE_HAND_REPORT_MD_PATH = REPORTS_DIR / "pose_hand_test.md"

SUPPORTED_VIDEO_EXTENSIONS = {".avi", ".mp4", ".mov", ".mkv", ".webm"}

# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
RANDOM_SEED = 42

# ---------------------------------------------------------------------------
# Sampling configuration
# ---------------------------------------------------------------------------
# Candidate HMDB51 classes where human body/hand motion is likely to be visible.
# These are examples only — the actual selection below is restricted to classes
# that exist in Notebook 01's class mapping (and on disk).
PREFERRED_SAMPLE_CLASSES = [
    "clap", "drink", "eat", "catch", "throw", "kick",
    "punch", "push", "pick", "pour", "wave", "golf",
]

NUM_CLASSES_TO_SAMPLE = 6      # how many classes to actually test
NUM_VIDEOS_PER_CLASS = 1       # videos sampled per selected class
FRAMES_PER_VIDEO = 2           # representative frames extracted per video

# ---------------------------------------------------------------------------
# Pose model configuration (MediaPipe Pose, static-image mode since we test
# independent frames rather than a continuous video stream)
# ---------------------------------------------------------------------------
POSE_MIN_DETECTION_CONFIDENCE = 0.5
POSE_MIN_TRACKING_CONFIDENCE = 0.5

# ---------------------------------------------------------------------------
# Hand model configuration (MediaPipe Hands, static-image mode)
# ---------------------------------------------------------------------------
HAND_MIN_DETECTION_CONFIDENCE = 0.5
HAND_MIN_TRACKING_CONFIDENCE = 0.5
MAX_NUM_HANDS = 2

# ---------------------------------------------------------------------------
# Visualization
# ---------------------------------------------------------------------------
MAX_VISUALIZED_FRAMES = 6

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("DATASET_ROOT              :", DATASET_ROOT)
print("TRAINING_ROOT               :", TRAINING_ROOT)
print("CLASS_MAPPING_PATH            :", CLASS_MAPPING_PATH)
print("POSE_HAND_REPORT_JSON_PATH      :", POSE_HAND_REPORT_JSON_PATH)
print("POSE_HAND_REPORT_MD_PATH       :", POSE_HAND_REPORT_MD_PATH)
print(f"Sampling up to {NUM_CLASSES_TO_SAMPLE} classes, "
      f"{NUM_VIDEOS_PER_CLASS} video(s)/class, {FRAMES_PER_VIDEO} frame(s)/video")
print(f"Pose confidence: detection={POSE_MIN_DETECTION_CONFIDENCE}, "
      f"tracking={POSE_MIN_TRACKING_CONFIDENCE}")
print(f"Hand confidence: detection={HAND_MIN_DETECTION_CONFIDENCE}, "
      f"tracking={HAND_MIN_TRACKING_CONFIDENCE}, max_num_hands={MAX_NUM_HANDS}")


DATASET_ROOT              : /content/drive/MyDrive/Technical-Projects/SIH26174/training/data/raw/hmdb51_sta
TRAINING_ROOT               : /content/drive/MyDrive/Technical-Projects/SIH26174/training
CLASS_MAPPING_PATH            : /content/drive/MyDrive/Technical-Projects/SIH26174/training/configs/action_classes.json
POSE_HAND_REPORT_JSON_PATH      : /content/drive/MyDrive/Technical-Projects/SIH26174/training/reports/pose_hand_test.json
POSE_HAND_REPORT_MD_PATH       : /content/drive/MyDrive/Technical-Projects/SIH26174/training/reports/pose_hand_test.md
Sampling up to 6 classes, 1 video(s)/class, 2 frame(s)/video
Pose confidence: detection=0.5, tracking=0.5
Hand confidence: detection=0.5, tracking=0.5, max_num_hands=2


## 4. Mount Google Drive

Mounts Drive when running in Colab; falls back gracefully outside Colab as long as `DATASET_ROOT`
is reachable another way.


In [4]:
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print(
        "Not running inside Google Colab — skipping drive.mount(). "
        "Make sure DATASET_ROOT is reachable on this filesystem."
    )


Mounted at /content/drive


## 5. Verify Dataset

Confirm the HMDB51 dataset root exists. The dataset is treated strictly as read-only source
material — nothing here is renamed, moved, or modified.


In [5]:
if not DATASET_ROOT.exists() or not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f"Dataset root does not exist or is not a directory:\n  {DATASET_ROOT}\n"
        "Make sure Google Drive is mounted and the HMDB51 dataset has been extracted here."
    )

print("Dataset root OK:", DATASET_ROOT)


Dataset root OK: /content/drive/MyDrive/Technical-Projects/SIH26174/training/data/raw/hmdb51_sta


## 6. Load Class Mapping

Load the class mapping produced by **Notebook 01**. Only classes present in this mapping are
eligible for sampling — the notebook does not invent or assume classes beyond what Notebook 01
already established.


In [6]:
if not CLASS_MAPPING_PATH.exists():
    raise FileNotFoundError(
        f"Expected Notebook 01 output not found:\n  {CLASS_MAPPING_PATH}\n"
        "Run 01_dataset_inspection.ipynb first."
    )

with open(CLASS_MAPPING_PATH, "r") as f:
    id_to_class = json.load(f)

mapped_class_names = sorted(id_to_class.values())
print(f"Loaded {len(mapped_class_names)} classes from: {CLASS_MAPPING_PATH}")
print("First 5 classes:", mapped_class_names[:5])


Loaded 51 classes from: /content/drive/MyDrive/Technical-Projects/SIH26174/training/configs/action_classes.json
First 5 classes: ['brush_hair', 'cartwheel', 'catch', 'chew', 'clap']


## 7. Select Representative Sample

Deterministically select a small set of HMDB51 videos to test against. We start from
`PREFERRED_SAMPLE_CLASSES` — action classes where body/hand motion is more likely to be visible —
but only keep classes that are present in the Notebook 01 class mapping **and** actually have a
folder on disk. If fewer than `NUM_CLASSES_TO_SAMPLE` preferred classes are available, we top up
with other seeded-random mapped classes.


In [7]:
def find_video_files(class_dir: Path):
    if not class_dir.is_dir():
        return []
    return sorted(
        p for p in class_dir.iterdir()
        if p.is_file() and p.suffix.lower() in SUPPORTED_VIDEO_EXTENSIONS
    )


classes_with_folder = [c for c in mapped_class_names if (DATASET_ROOT / c).is_dir()]
missing_on_disk = [c for c in mapped_class_names if c not in classes_with_folder]
if missing_on_disk:
    print(f"NOTE: {len(missing_on_disk)} mapped class(es) have no folder on disk and are "
          f"excluded from sampling: {missing_on_disk}")

available_preferred = [c for c in PREFERRED_SAMPLE_CLASSES if c in classes_with_folder]
missing_preferred = [c for c in PREFERRED_SAMPLE_CLASSES if c not in classes_with_folder]
if missing_preferred:
    print(f"NOTE: {len(missing_preferred)} preferred class(es) not usable in this dataset copy "
          f"and will be skipped: {missing_preferred}")

rng = random.Random(RANDOM_SEED)

selected_classes = available_preferred[:NUM_CLASSES_TO_SAMPLE]
if len(selected_classes) < NUM_CLASSES_TO_SAMPLE:
    remaining_pool = [c for c in classes_with_folder if c not in selected_classes]
    rng.shuffle(remaining_pool)
    needed = NUM_CLASSES_TO_SAMPLE - len(selected_classes)
    selected_classes.extend(remaining_pool[:needed])

selected_classes = sorted(selected_classes)
print(f"Selected {len(selected_classes)} classes for this test:")
print(" ", selected_classes)


Selected 6 classes for this test:
  ['catch', 'clap', 'drink', 'eat', 'kick', 'throw']


In [8]:
sample_video_records = []  # {video_path: Path, class_name: str}

for class_name in selected_classes:
    class_dir = DATASET_ROOT / class_name
    videos = find_video_files(class_dir)
    if not videos:
        print(f"WARNING: no video files found for class '{class_name}' — skipping.")
        continue

    chosen = rng.sample(videos, min(NUM_VIDEOS_PER_CLASS, len(videos)))
    for video_path in chosen:
        sample_video_records.append({"video_path": video_path, "class_name": class_name})

print(f"Total sample videos selected: {len(sample_video_records)}")
for rec in sample_video_records:
    print(f"  [{rec['class_name']}] {rec['video_path'].name}")


Total sample videos selected: 6
  [catch] Torwarttraining_catch_f_cm_np1_le_bad_11.avi
  [clap] Deutscher_Klatsch_-_The_Clap_-_Lexy_clap_f_cm_np1_fr_bad_10.avi
  [drink] AMADEUS_drink_u_nm_np1_fr_goo_11.avi
  [eat] The_Departed_-_Part_1_eat_u_nm_np1_fr_goo_1.avi
  [kick] Veoh_Alpha_Dog_1_kick_f_nm_np1_ba_med_48.avi
  [throw] Compilationknifethrowing_throw_u_nm_np1_ba_med_9.avi


## 8. Extract Representative Frames

For each selected video, extract a small number of evenly spaced frames (`FRAMES_PER_VIDEO`)
rather than scanning the whole video. Unreadable videos are reported and skipped, not treated as a
fatal error.


In [9]:
def extract_representative_frames(video_path: Path, num_frames: int):
    """Return up to num_frames evenly spaced BGR frames from a video, or [] on failure."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        return []

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count <= 0:
        cap.release()
        return []

    if num_frames == 1:
        indices = [frame_count // 2]
    else:
        indices = np.linspace(0, frame_count - 1, num=num_frames)
        indices = sorted(set(np.round(indices).astype(int).tolist()))

    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if ok and frame is not None:
            frames.append({"frame_index": int(idx), "frame": frame})

    cap.release()
    return frames


test_frames = []  # {video_path, class_name, frame_index, frame (BGR np.ndarray)}
unreadable_sample_videos = []

for rec in sample_video_records:
    frames = extract_representative_frames(rec["video_path"], FRAMES_PER_VIDEO)
    if not frames:
        unreadable_sample_videos.append(str(rec["video_path"]))
        continue
    for item in frames:
        test_frames.append({
            "video_path": rec["video_path"],
            "class_name": rec["class_name"],
            "frame_index": item["frame_index"],
            "frame": item["frame"],
        })

print(f"Representative frames extracted: {len(test_frames)}")
if unreadable_sample_videos:
    print(f"WARNING: {len(unreadable_sample_videos)} sample video(s) could not be read:")
    for v in unreadable_sample_videos:
        print("  -", v)


Representative frames extracted: 6


## 9. Initialize Pretrained Pose Model

MediaPipe Pose is initialized in static-image mode (each frame is treated independently, since our
frames come from unrelated sampled videos rather than a continuous stream).


In [10]:
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

pose_model = mp_pose.Pose(
    static_image_mode=True,
    min_detection_confidence=POSE_MIN_DETECTION_CONFIDENCE,
    min_tracking_confidence=POSE_MIN_TRACKING_CONFIDENCE,
)

print("MediaPipe Pose initialized (static_image_mode=True).")


AttributeError: module 'mediapipe' has no attribute 'solutions'

## 10. Run Pose Estimation

Run pretrained pose estimation on every extracted frame. For each frame we record whether pose
landmarks were detected, how many landmarks were found, and the mean landmark visibility score
when available — plus per-frame inference timing.


In [ ]:
pose_results_records = []   # per-frame summary dicts
pose_raw_results = []       # {video_path, class_name, frame_index, frame, mp_result}
pose_timings = []

for item in test_frames:
    frame_bgr = item["frame"]
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    start = time.time()
    result = pose_model.process(frame_rgb)
    elapsed = time.time() - start
    pose_timings.append(elapsed)

    detected = result.pose_landmarks is not None
    landmark_count = len(result.pose_landmarks.landmark) if detected else 0
    mean_visibility = (
        float(np.mean([lm.visibility for lm in result.pose_landmarks.landmark]))
        if detected else None
    )

    pose_results_records.append({
        "source_video": str(item["video_path"]),
        "hmdb51_action_class": item["class_name"],
        "frame_index": item["frame_index"],
        "pose_detected": detected,
        "landmark_count": landmark_count,
        "mean_visibility": round(mean_visibility, 4) if mean_visibility is not None else None,
    })

    pose_raw_results.append({
        "video_path": item["video_path"],
        "class_name": item["class_name"],
        "frame_index": item["frame_index"],
        "frame_rgb": frame_rgb,
        "mp_result": result,
    })

pose_df = pd.DataFrame(pose_results_records)
frames_with_pose = int(pose_df["pose_detected"].sum()) if not pose_df.empty else 0

print(f"Frames tested          : {len(test_frames)}")
print(f"Frames with pose detected: {frames_with_pose}")
pose_df.head(10)


## 11. Visualize Pose Results

Show the pose skeleton drawn on a small subset of tested frames, so detections can be judged
visually.


In [ ]:
frames_to_show = pose_raw_results[:MAX_VISUALIZED_FRAMES]
num_show = len(frames_to_show)

if num_show == 0:
    print("No frames available to visualize.")
else:
    cols = min(3, num_show)
    rows = (num_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, item in zip(axes, frames_to_show):
        annotated = item["frame_rgb"].copy()
        result = item["mp_result"]
        if result.pose_landmarks is not None:
            mp_drawing.draw_landmarks(
                annotated,
                result.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style(),
            )
            title_suffix = "pose detected"
        else:
            title_suffix = "no pose detected"

        ax.imshow(annotated)
        ax.set_title(f"{item['class_name']} (frame {item['frame_index']}) — {title_suffix}", fontsize=9)
        ax.axis("off")

    for ax in axes[num_show:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


## 12. Initialize Pretrained Hand Model

MediaPipe Hands is initialized in static-image mode, matching the pose model's configuration.


In [ ]:
mp_hands = mp.solutions.hands

hand_model = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=MAX_NUM_HANDS,
    min_detection_confidence=HAND_MIN_DETECTION_CONFIDENCE,
    min_tracking_confidence=HAND_MIN_TRACKING_CONFIDENCE,
)

print(f"MediaPipe Hands initialized (static_image_mode=True, max_num_hands={MAX_NUM_HANDS}).")


## 13. Run Hand Estimation

Run pretrained hand estimation on the same frames. We record whether at least one hand was
detected and how many hands were found, plus per-frame inference timing. We do **not** assume
every frame contains a detectable hand — a frame with zero hands detected is a normal, expected
outcome, not an error.


In [ ]:
hand_results_records = []
hand_raw_results = []
hand_timings = []

for item in test_frames:
    frame_bgr = item["frame"]
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    start = time.time()
    result = hand_model.process(frame_rgb)
    elapsed = time.time() - start
    hand_timings.append(elapsed)

    hand_landmarks_list = result.multi_hand_landmarks or []
    num_hands = len(hand_landmarks_list)
    detected = num_hands > 0

    hand_results_records.append({
        "source_video": str(item["video_path"]),
        "hmdb51_action_class": item["class_name"],
        "frame_index": item["frame_index"],
        "hand_detected": detected,
        "num_hands_detected": num_hands,
    })

    hand_raw_results.append({
        "video_path": item["video_path"],
        "class_name": item["class_name"],
        "frame_index": item["frame_index"],
        "frame_rgb": frame_rgb,
        "mp_result": result,
    })

hand_df = pd.DataFrame(hand_results_records)
frames_with_hand = int(hand_df["hand_detected"].sum()) if not hand_df.empty else 0
total_hands_detected = int(hand_df["num_hands_detected"].sum()) if not hand_df.empty else 0

print(f"Frames tested             : {len(test_frames)}")
print(f"Frames with >=1 hand detected: {frames_with_hand}")
print(f"Total hands detected (sum)   : {total_hands_detected}")
hand_df.head(10)


## 14. Visualize Hand Results

Show hand landmarks drawn on a small subset of tested frames.


In [ ]:
frames_to_show = hand_raw_results[:MAX_VISUALIZED_FRAMES]
num_show = len(frames_to_show)

if num_show == 0:
    print("No frames available to visualize.")
else:
    cols = min(3, num_show)
    rows = (num_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, item in zip(axes, frames_to_show):
        annotated = item["frame_rgb"].copy()
        result = item["mp_result"]
        hand_landmarks_list = result.multi_hand_landmarks or []

        for hand_landmarks in hand_landmarks_list:
            mp_drawing.draw_landmarks(
                annotated,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS,
                landmark_drawing_spec=mp_drawing_styles.get_default_hand_landmarks_style(),
                connection_drawing_spec=mp_drawing_styles.get_default_hand_connections_style(),
            )

        title_suffix = f"{len(hand_landmarks_list)} hand(s)" if hand_landmarks_list else "no hand detected"
        ax.imshow(annotated)
        ax.set_title(f"{item['class_name']} (frame {item['frame_index']}) — {title_suffix}", fontsize=9)
        ax.axis("off")

    for ax in axes[num_show:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


## 15. Calculate Baseline Detection Statistics

Detection-availability rates for pose and hands. These are **not** accuracy metrics — they only
describe how often each model produced any landmarks at all on this small sample.


In [ ]:
num_frames_tested = len(test_frames)

pose_detection_rate = (frames_with_pose / num_frames_tested) if num_frames_tested else 0.0
hand_detection_rate = (frames_with_hand / num_frames_tested) if num_frames_tested else 0.0

print(f"Frames tested            : {num_frames_tested}")
print(f"Pose detections (frames)  : {frames_with_pose}")
print(f"Pose detection rate        : {pose_detection_rate:.0%}")
print(f"Hand detections (frames)    : {frames_with_hand}")
print(f"Total hands detected (sum)   : {total_hands_detected}")
print(f"Hand detection rate            : {hand_detection_rate:.0%}")

if not pose_df.empty:
    pose_by_class = pose_df.groupby("hmdb51_action_class")["pose_detected"].mean().round(2)
    print("\nPose detection rate by sampled class:")
    print(pose_by_class)

if not hand_df.empty:
    hand_by_class = hand_df.groupby("hmdb51_action_class")["hand_detected"].mean().round(2)
    print("\nHand detection rate by sampled class:")
    print(hand_by_class)


## 16. Measure Lightweight Inference Timing

A small, indicative timing measurement over the tested frames only — not a formal benchmark.


In [ ]:
total_pose_seconds = float(sum(pose_timings))
avg_pose_seconds = (total_pose_seconds / num_frames_tested) if num_frames_tested else 0.0
approx_pose_fps = (1.0 / avg_pose_seconds) if avg_pose_seconds > 0 else 0.0

total_hand_seconds = float(sum(hand_timings))
avg_hand_seconds = (total_hand_seconds / num_frames_tested) if num_frames_tested else 0.0
approx_hand_fps = (1.0 / avg_hand_seconds) if avg_hand_seconds > 0 else 0.0

print(f"Frames tested                 : {num_frames_tested}")
print(f"Total pose inference time      : {total_pose_seconds:.3f}s")
print(f"Average pose inference time     : {avg_pose_seconds * 1000:.1f} ms/frame")
print(f"Approximate pose FPS              : {approx_pose_fps:.1f}")
print(f"Total hand inference time          : {total_hand_seconds:.3f}s")
print(f"Average hand inference time         : {avg_hand_seconds * 1000:.1f} ms/frame")
print(f"Approximate hand FPS                  : {approx_hand_fps:.1f}")
print("(Note: timings can include one-off model warm-up; treat this as indicative only.)")


## 17. Record Limitations

An honest, observation-based assessment derived only from the results collected above. Nothing
here is fabricated, and detection rate alone is not treated as proof of accuracy.


In [ ]:
limitation_notes = []

limitation_notes.append(
    "This test used MediaPipe's pretrained Pose and Hands solutions on a small, deterministic "
    f"sample of {num_frames_tested} HMDB51 frames drawn from {len(selected_classes)} action "
    "classes. It does not constitute a full accuracy evaluation."
)
limitation_notes.append(
    f"Pose landmarks were detected in {frames_with_pose}/{num_frames_tested} frames "
    f"({pose_detection_rate:.0%}). This is a detection-availability rate, not an accuracy score — "
    "a detected skeleton may still be imprecise, and the visualizations in Section 11 should be "
    "reviewed manually."
)
limitation_notes.append(
    f"At least one hand was detected in {frames_with_hand}/{num_frames_tested} frames "
    f"({hand_detection_rate:.0%}), with {total_hands_detected} hand instance(s) detected in "
    "total. Hands are frequently smaller, faster-moving, and more prone to occlusion than the "
    "body, so a lower hand detection rate than pose detection rate is expected and not "
    "necessarily a model failure."
)

no_pose_classes = sorted(pose_df.loc[~pose_df["pose_detected"], "hmdb51_action_class"].unique().tolist()) if not pose_df.empty else []
if no_pose_classes:
    limitation_notes.append(
        f"Classes with at least one frame showing no detected pose: {no_pose_classes}. "
        "Possible causes include partial/cropped body framing, unusual body orientation, motion "
        "blur, or a person not being clearly visible in the sampled frame."
    )

no_hand_classes = sorted(hand_df.loc[~hand_df["hand_detected"], "hmdb51_action_class"].unique().tolist()) if not hand_df.empty else []
if no_hand_classes:
    limitation_notes.append(
        f"Classes with at least one frame showing no detected hand: {no_hand_classes}. "
        "Possible causes include hands being out of frame, occluded by the body or an object, "
        "or too small/blurred at the sampled resolution."
    )

limitation_notes.append(
    "HMDB51 does not represent the eventual SIH laboratory experiment's camera framing, lighting, "
    "payload-rack context, or astronaut body orientation (including microgravity-specific "
    "orientations). Pose/hand behavior on the real experiment setup cannot be inferred from this "
    "HMDB51 sample alone."
)
limitation_notes.append(
    "This notebook uses MediaPipe's classic 'solutions' API in static-image mode. No custom "
    "pose or hand model was trained, and no experiment-specific landmarks were defined."
)

print("Observed limitations:")
for note in limitation_notes:
    print(" -", note)


## 18. Save Report

Save a machine-readable report to `training/reports/pose_hand_test.json`, and a human-readable
version to `training/reports/pose_hand_test.md`.

Per the project's testing-notebook convention, this report does **not** claim the pretrained
pose/hand models are "sufficient" or "insufficient" for the final SIH laboratory experiment —
HMDB51 is not that experiment's dataset, and this test covers only a small sample.


In [ ]:
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_STATUS = "PRETRAINED POSE/HAND BASELINE TEST COMPLETE"
SUITABILITY_NOTE = (
    "Suitability for the final SIH laboratory experiment cannot yet be definitively determined "
    "from HMDB51 alone, because HMDB51 is an action-recognition dataset (not a laboratory "
    "pose/hand dataset) and this notebook evaluates only a small representative sample."
)

report = {
    "notebook": "training/notebooks/04_pose_hand_testing.ipynb",
    "models": {
        "pose_model": "MediaPipe Pose (solutions API, static_image_mode=True)",
        "hand_model": "MediaPipe Hands (solutions API, static_image_mode=True)",
        "mediapipe_version": mp.__version__,
        "pose_min_detection_confidence": POSE_MIN_DETECTION_CONFIDENCE,
        "pose_min_tracking_confidence": POSE_MIN_TRACKING_CONFIDENCE,
        "hand_min_detection_confidence": HAND_MIN_DETECTION_CONFIDENCE,
        "hand_min_tracking_confidence": HAND_MIN_TRACKING_CONFIDENCE,
        "max_num_hands": MAX_NUM_HANDS,
    },
    "sampling": {
        "random_seed": RANDOM_SEED,
        "selected_classes": selected_classes,
        "num_sample_videos": len(sample_video_records),
        "frames_per_video": FRAMES_PER_VIDEO,
        "num_frames_tested": num_frames_tested,
        "unreadable_sample_videos": unreadable_sample_videos,
    },
    "pose_results": {
        "frames_with_pose_detected": frames_with_pose,
        "pose_detection_rate": round(pose_detection_rate, 4),
        "per_frame": pose_results_records,
    },
    "hand_results": {
        "frames_with_hand_detected": frames_with_hand,
        "hand_detection_rate": round(hand_detection_rate, 4),
        "total_hands_detected": total_hands_detected,
        "per_frame": hand_results_records,
    },
    "timing": {
        "pose": {
            "total_seconds": round(total_pose_seconds, 4),
            "avg_seconds_per_frame": round(avg_pose_seconds, 4),
            "approx_fps": round(approx_pose_fps, 2),
        },
        "hand": {
            "total_seconds": round(total_hand_seconds, 4),
            "avg_seconds_per_frame": round(avg_hand_seconds, 4),
            "approx_fps": round(approx_hand_fps, 2),
        },
    },
    "limitations": limitation_notes,
    "baseline_status": BASELINE_STATUS,
    "suitability_note": SUITABILITY_NOTE,
}

with open(POSE_HAND_REPORT_JSON_PATH, "w") as f:
    json.dump(report, f, indent=2)

print("Saved JSON report ->", POSE_HAND_REPORT_JSON_PATH)


In [ ]:
md_lines = []
md_lines.append("# Pretrained Pose / Hand Estimation Baseline Test\n\n")
md_lines.append(f"- **Pose model:** MediaPipe Pose (solutions API, static_image_mode=True)\n")
md_lines.append(f"- **Hand model:** MediaPipe Hands (solutions API, static_image_mode=True)\n")
md_lines.append(f"- **MediaPipe version:** {mp.__version__}\n")
md_lines.append(f"- **Frames tested:** {num_frames_tested}\n")
md_lines.append(f"- **Pose detections:** {frames_with_pose}/{num_frames_tested} "
                 f"({pose_detection_rate:.0%})\n")
md_lines.append(f"- **Hand detections:** {frames_with_hand}/{num_frames_tested} "
                 f"({hand_detection_rate:.0%}), {total_hands_detected} hand instance(s) total\n")
md_lines.append(f"- **Approx. pose FPS:** {approx_pose_fps:.1f}\n")
md_lines.append(f"- **Approx. hand FPS:** {approx_hand_fps:.1f}\n\n")

md_lines.append("## Limitations\n\n")
for note in limitation_notes:
    md_lines.append(f"- {note}\n")

md_lines.append("\n## Baseline Status\n\n")
md_lines.append(f"**{BASELINE_STATUS}**\n\n")
md_lines.append(f"{SUITABILITY_NOTE}\n")

with open(POSE_HAND_REPORT_MD_PATH, "w") as f:
    f.writelines(md_lines)

print("Saved Markdown report ->", POSE_HAND_REPORT_MD_PATH)


## 19. Final Summary

Consolidated output. This notebook reports pose/hand detection observations on a small HMDB51
sample — it does not claim the models are sufficient or insufficient for the final SIH laboratory
experiment.


In [ ]:
print("=" * 70)
print("NOTEBOOK 04 — POSE / HAND TESTING — FINAL SUMMARY")
print("=" * 70)
print(f"Dataset                    : HMDB51 ({DATASET_ROOT})")
print(f"Pose model                  : MediaPipe Pose (solutions API)")
print(f"Hand model                   : MediaPipe Hands (solutions API)")
print(f"Frames tested                  : {num_frames_tested}")
print(f"Pose detections                  : {frames_with_pose}")
print(f"Pose detection rate                : {pose_detection_rate:.0%}")
print(f"Hand detections                      : {frames_with_hand}")
print(f"Hand detection rate                    : {hand_detection_rate:.0%}")
print(f"Pose average inference time              : {avg_pose_seconds * 1000:.1f} ms/frame")
print(f"Hand average inference time                : {avg_hand_seconds * 1000:.1f} ms/frame")
print(f"Baseline status                              :")
print(f"    {BASELINE_STATUS}")
print(f"Suitability note                               :")
print(f"    {SUITABILITY_NOTE}")
print("=" * 70)
print("Reminder: no pose or hand model was trained or fine-tuned in this notebook.")
print("No action-recognition model was loaded, trained, or run in this notebook.")
print("Generated files:")
print(f"  - {POSE_HAND_REPORT_JSON_PATH}")
print(f"  - {POSE_HAND_REPORT_MD_PATH}")
